-character level language model
-trained data of characters
-at test time  keeps predicting next charcter

-ex:
-  suresh
-   ramesh
-   sunil
-   suraj
-   naresh
-   here might predict .sh.

-let the menatl model for now be each name is sentence each charater is a word

-in n gram models
-given a sentence p(w1, w2, w3, w4) = p(w1)p(w2|w1)p(w3|w1,w2)p(w4|w1,w2,w3)
-similarly for a given 


For a batch size of $N$ and $C$ classes, the multi-class cross-entropy loss is $\mathcal{L}_{\text{batch}} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log(p_{i,c})$.

https://share.gemini.google/1g2DMTImQ4bE

An **$n$-gram model** is a classical statistical language model that predicts the next word by looking back at only the preceding $n-1$ words, estimating probabilities purely through frequency counts.

It directly addresses the probability chain rule discussed earlier, simplifying it using the **Markov assumption**.

---

### 1. From the Chain Rule to the Markov Assumption

Recall the exact chain rule of language modeling:

$$P(w_1, w_2, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, w_2, \dots, w_{t-1})$$

In a 50-word sentence, calculating the 50th word requires conditioning on a 49-word ordered tuple:

$$P(w_{50} \mid w_1, w_2, \dots, w_{49})$$

In traditional statistics, you cannot estimate this probability. That specific 49-word sequence has almost certainly never appeared before in human writing, meaning its count in your training data is 0.

#### The $n$-Gram Approximation

An $n$-gram model discards distant history, assuming that the probability of the current word depends **only on the most recent $n-1$ words**:

$$P(w_t \mid w_1, w_2, \dots, w_{t-1}) \approx P(w_t \mid w_{t-n+1}, \dots, w_{t-1})$$

| Model | $n$ | Looks Back | Mathematical Form |
| --- | --- | --- | --- |
| **Unigram** | 1 | 0 words (bag of words) | $P(w_t)$ |
| **Bigram** | 2 | 1 word | $P(w_t \mid w_{t-1})$ |
| **Trigram** | 3 | 2 words | $P(w_t \mid w_{t-2}, w_{t-1})$ |
| **4-gram** | 4 | 3 words | $P(w_t \mid w_{t-3}, w_{t-2}, w_{t-1})$ |

Notice that the condition $(w_{t-n+1}, \dots, w_{t-1})$ remains an **ordered tuple**. A bigram model treats $(w_1 = \text{"dog"}, w_2 = \text{"barks"})$ as completely distinct from $(w_1 = \text{"barks"}, w_2 = \text{"dog"})$.

---

### 2. How an $n$-Gram Model "Trains" (Counting MLE)

Unlike neural networks, which use gradient descent and backpropagation, an $n$-gram model trains via direct **frequency counting** using Maximum Likelihood Estimation (MLE):

$$P_{\text{MLE}}(w_t \mid w_{t-n+1}, \dots, w_{t-1}) = \frac{\text{Count}(w_{t-n+1}, \dots, w_{t-1}, w_t)}{\text{Count}(w_{t-n+1}, \dots, w_{t-1})}$$

#### Concrete Example (Bigram Model)

Suppose our entire training corpus consists of three short sentences:

1. `<s> the cat sat on the mat </s>`
2. `<s> the dog sat on the rug </s>`
3. `<s> the cat ate the food </s>`

*(where `<s>` and `</s>` mark the start and end of a sentence)*

To calculate $P(\text{"sat"} \mid \text{"cat"})$:

1. Count how many times the prefix `"cat"` appears: $\text{Count}(\text{"cat"}) = 2$.
2. Count how many times the bigram `("cat", "sat")` appears: $\text{Count}(\text{"cat", "sat"}) = 1$.
3. Divide:

$$P(\text{"sat"} \mid \text{"cat"}) = \frac{\text{Count}(\text{"cat", "sat"})}{\text{Count}(\text{"cat"})} = \frac{1}{2} = 0.5$$



To calculate $P(\text{"ate"} \mid \text{"cat"})$:


$$P(\text{"ate"} \mid \text{"cat"}) = \frac{\text{Count}(\text{"cat", "ate"})}{\text{Count}(\text{"cat"})} = \frac{1}{2} = 0.5$$

---

### 3. The Three Structural Breakdowns of $n$-Grams

While computationally trivial to train, $n$-gram models hit three severe mathematical ceilings that neural networks eventually solved:

#### A. The Curse of Dimensionality

If your vocabulary has $\vert{}V\vert{} = 50,000$ words:

* A bigram model must track a table of $\vert{}V\vert{}^2 = 2.5 \times 10^9$ possible pairs.
* A trigram model requires $\vert{}V\vert{}^3 = 1.25 \times 10^{14}$ possible triplets.
* A 5-gram model requires $\vert{}V\vert{}^5 \approx 3.1 \times 10^{23}$ values (more than all storage on Earth).

Because memory requirements scale exponentially as $O(\vert{}V\vert{}^n)$, $n$-gram models rarely scaled beyond $n = 5$.

#### B. The Zero-Probability / Sparsity Problem

If the test sentence is `"the dog ate the bone"`, but the training corpus never saw the bigram `("dog", "ate")`:

$$\text{Count}(\text{"dog", "ate"}) = 0 \implies P(\text{"ate"} \mid \text{"dog"}) = 0$$

Because the total sentence probability is a product of probabilities:

$$P(\text{Sentence}) = P(\text{"dog"} \mid \text{"the"}) \times 0 \times \dots = 0$$

A single novel word pair causes the entire sentence probability to crash to zero. Researchers had to invent complex heuristic patches called **smoothing** (Laplace, Good-Turing, and Kneser-Ney smoothing) to steal small fractions of probability mass from frequent words and assign them to unseen combinations.

#### C. Zero Semantic Generalization

In an $n$-gram model, words are discrete, orthogonal symbols (atomic indices):

```text
"cat" = Token #1042
"feline" = Token #8911
"dog" = Token #2051

```

If the model has seen `"the cat drank the milk"` 10,000 times, but has never seen `"the feline drank the milk"`, it assigns $0$ probability to the second sentence. It has no mathematical mechanism to recognize that a *cat* and a *feline* share behavioral or grammatical properties.

---

### How Transformers Compare to $n$-Grams

| Property | $n$-Gram Models | Modern Transformers |
| --- | --- | --- |
| **History Window ($n$)** | Fixed and small ($n \le 5$) | Massive ($2,048$ to $1,000,000+$ tokens) |
| **Parameter Scaling** | Exponential: $O(\Vert{}V\Vert{}^n)$ | Linear/Quadratic with dimension & context |
| **Word Representation** | Discrete strings / IDs | Continuous vectors (embeddings) |
| **Generalization** | Exact-match counting only | Semantic similarity via dot products |
| **Unseen Phrasing** | Assigns zero (needs smoothing) | Generalizes naturally via vector proximity |

In [1]:
words = open("./names.txt", "r").read().splitlines()

In [2]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
print(min(len(w) for w in words))
print(max(len(w) for w in words))

2
15


In [4]:
for word in words[:1]:
    for i in range(len(word)):
        if i < (len(word)-1):
            print(word[i],word[i+1])

e m
m m
m a


In [5]:
for word in words[:1]:
    for i,j in zip(word,word[1:]):
        print(i,j)

e m
m m
m a


ther is more info we are missing<br>
not accounting for the starting and ending letter likelihood<br>
so

In [6]:
for word in words[:1]:
    chars = ["<S>"] + list(word)  + ["<E>"]
    for ch1, ch2 in zip(chars, chars[1:]):
        print(ch1, ch2)

<S> e
e m
m m
m a
a <E>


In [7]:
chars = sorted(list(set("".join(words))))
print(chars)
print(len(chars))

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
26


In [8]:
ctoi = {}
for i,c in enumerate(chars, start=1):
    ctoi[c] = i
ctoi["."] = 0
ctoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [9]:
import torch

In [10]:
N = torch.zeros((27, 27), dtype = torch.int32)

In [11]:
for word in words:
    chars = ["."] + list(word) + ["."]
    for ch1, ch2 in zip(chars, chars[1:]):
        N[ctoi[ch1], ctoi[ch2]] += 1

In [12]:
N

tensor([[   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
         1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
          134,  535,  929],
        [6640,  556,  541,  470, 1042,  692,  134,  168, 2332, 1650,  175,  568,
         2528, 1634, 5438,   63,   82,   60, 3264, 1118,  687,  381,  834,  161,
          182, 2050,  435],
        [ 114,  321,   38,    1,   65,  655,    0,    0,   41,  217,    1,    0,
          103,    0,    4,  105,    0,    0,  842,    8,    2,   45,    0,    0,
            0,   83,    0],
        [  97,  815,    0,   42,    1,  551,    0,    2,  664,  271,    3,  316,
          116,    0,    0,  380,    1,   11,   76,    5,   35,   35,    0,    0,
            3,  104,    4],
        [ 516, 1303,    1,    3,  149, 1283,    5,   25,  118,  674,    9,    3,
           60,   30,   31,  378,    0,    1,  424,   29,    4,   92,   17,   23,
            0,  317,    1],
        [3983,  679,  121,  153,  384, 1271,   82,

In [13]:
# let's convert each row of N into a probability distribution
P = N.float() / N.sum(dim=1, keepdim=True, dtype= torch.float32) # (27, 27) / (27,1) => (27, 27)
print(P)

tensor([[0.0000e+00, 1.3767e-01, 4.0770e-02, 4.8138e-02, 5.2758e-02, 4.7794e-02,
         1.3018e-02, 2.0885e-02, 2.7284e-02, 1.8450e-02, 7.5610e-02, 9.2498e-02,
         4.9074e-02, 7.9231e-02, 3.5776e-02, 1.2300e-02, 1.6077e-02, 2.8720e-03,
         5.1166e-02, 6.4153e-02, 4.0833e-02, 2.4350e-03, 1.1738e-02, 9.5839e-03,
         4.1832e-03, 1.6702e-02, 2.9001e-02],
        [1.9596e-01, 1.6408e-02, 1.5966e-02, 1.3870e-02, 3.0751e-02, 2.0422e-02,
         3.9546e-03, 4.9579e-03, 6.8821e-02, 4.8694e-02, 5.1645e-03, 1.6763e-02,
         7.4605e-02, 4.8222e-02, 1.6048e-01, 1.8592e-03, 2.4199e-03, 1.7707e-03,
         9.6326e-02, 3.2994e-02, 2.0274e-02, 1.1244e-02, 2.4613e-02, 4.7514e-03,
         5.3711e-03, 6.0499e-02, 1.2838e-02],
        [4.3100e-02, 1.2136e-01, 1.4367e-02, 3.7807e-04, 2.4575e-02, 2.4764e-01,
         0.0000e+00, 0.0000e+00, 1.5501e-02, 8.2042e-02, 3.7807e-04, 0.0000e+00,
         3.8941e-02, 0.0000e+00, 1.5123e-03, 3.9698e-02, 0.0000e+00, 0.0000e+00,
         3.1834e-

In [14]:
itoc = {i:c for c,i in ctoi.items()}
print(itoc)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [15]:
print(P[15])
print(P[15].max())
print(P[15].argmax())


tensor([0.1078, 0.0188, 0.0176, 0.0144, 0.0239, 0.0166, 0.0043, 0.0055, 0.0216,
        0.0087, 0.0020, 0.0086, 0.0780, 0.0329, 0.3039, 0.0145, 0.0120, 0.0004,
        0.1335, 0.0635, 0.0149, 0.0347, 0.0222, 0.0144, 0.0057, 0.0130, 0.0068])
tensor(0.3039)
tensor(14)


In [16]:
#sampling
g = torch.Generator().manual_seed(123)
for i in range(5):
    idx = 0
    name = ""
    while True:
        idx = torch.multinomial(P[idx], num_samples=1, replacement=True, generator= g).item()
        if idx == 0:
            break
        char = itoc[idx]
        name += char

    print(name)

ayl
eeiey
viannalizyr
pidouyra
jon


In [17]:
from collections import defaultdict

In [18]:
print(f"{ctoi=}")
print(f"{itoc=}")

ctoi={'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
itoc={1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [19]:
quadgram = defaultdict(lambda : torch.zeros((27,), dtype= torch.int64))
for word in words:
    chars = list("...") + list(word) + list(".")
    for i in range(len(chars) -3):
          quadgram[("".join(chars[i:i+3]))][ctoi[chars[i+3]]] += 1

In [20]:
#sampling trigram
g = torch.Generator().manual_seed(123)


for _ in range(5):
    key = "..."
    name = ""
    while True:
        next_ch_cnts = quadgram[key]
        # p = (next_ch_cnts + 1) / (next_ch_cnts.sum() + 27) # with smoothing (not working good)
        p = next_ch_cnts / next_ch_cnts.sum() # without smoothing
        idx = torch.multinomial(p, num_samples = 1, replacement = True, generator= g).item()
        if idx == 0:
            break
        next_char = itoc[idx]
        name += next_char
        key = key[1:] + next_char
    print(name)



aylan
jaydona
lawaz
raelynn
rai


In [21]:
Psmooth =  (N + 1) / N.sum(dim=1, keepdim= True)

In [22]:
#log likelihood of a given word in using our bigrad model
nll = 0.0
n = 0
for word in ["andrejq"]:
    chars = ["."] +  list(word) + ["."]
    for ch1, ch2 in zip(chars, chars[1:]):
        idx1 = ctoi[ch1]
        idx2 = ctoi[ch2]
        log_porb = torch.log(Psmooth[idx1, idx2])
        nll += -log_porb
        n += 1
        print(ch1+ch2, ":", f"{log_porb.item()=}")
print((nll/n).item())


.a : log_porb.item()=-1.9826651811599731
an : log_porb.item()=-1.8293771743774414
nd : log_porb.item()=-3.2579329013824463
dr : log_porb.item()=-2.5596866607666016
re : log_porb.item()=-2.012151002883911
ej : log_porb.item()=-5.899065017700195
jq : log_porb.item()=-7.972465991973877
q. : log_porb.item()=-2.238506317138672
3.4689812660217285


In [23]:
torch.tensor([10])/torch.tensor([4])

tensor([2.5000])

In [24]:
#modelling  using  a nerual network(bigram)
#a 27 dimensional one hot encoded vector representing prefix/context character
# 27 neurons where each neruron represents next charater 
# 27 weights of each neruron represent a weight associated with 27 possible characters
# so when one charater is given input to all neruons they multiply  give out their weight associated with that particular charater
# we sofmax these lgits to get  probabilites and and perform nll with true out

#prepare data
xs = []
ys = []
for word in words:
    chars = ["."] + list(word) + ["."]
    for ch1, ch2 in zip(chars, chars[1:]):
        xs.append(ctoi[ch1])
        ys.append(ctoi[ch2])

X = torch.tensor(xs)
Y = torch.tensor(ys)
W = torch.randn((27, 27), requires_grad= True)

In [25]:
# assume each column of W is a neuron and each row of that column represent it's weights
#we need to make each row of X into a one hot vecotr
import torch.nn.functional as F
X = F.one_hot(X, num_classes = 27)

In [26]:
print(X[:5])

tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0]])


In [27]:
print(f"{X.shape=}")
print(f"{W.shape=}")
print(f"{Y.shape=}")

X.shape=torch.Size([228146, 27])
W.shape=torch.Size([27, 27])
Y.shape=torch.Size([228146])


In [32]:
for i in range(10000):
    logits = X.float() @ W
    e_pow_logits = torch.exp(logits)
    Probs = e_pow_logits / e_pow_logits.sum(dim = 1, keepdim= True)
    loss = -torch.log(Probs[torch.arange(Probs.size(0)), Y]).mean()
    print(loss.item())
    loss.backward()
    # print(W.grad)
    with torch.no_grad():
        W -= 1 * W.grad
        W.grad = None

3.7623846530914307
3.7539427280426025
3.7455978393554688
3.7373459339141846
3.729184865951538
3.721113443374634
3.7131288051605225
3.705228805541992
3.6974120140075684
3.689676284790039
3.6820199489593506
3.674440860748291
3.666938304901123
3.659510612487793
3.652155637741089
3.6448724269866943
3.6376590728759766
3.6305158138275146
3.6234400272369385
3.6164309978485107
3.609487533569336
3.6026086807250977
3.5957934856414795
3.589040517807007
3.5823490619659424
3.5757181644439697
3.5691475868225098
3.562635898590088
3.5561816692352295
3.5497846603393555
3.543444871902466
3.537160634994507
3.530931234359741
3.52475643157959
3.5186350345611572
3.5125672817230225
3.506551504135132
3.5005877017974854
3.4946749210357666
3.488813638687134
3.483001947402954
3.4772398471832275
3.471527099609375
3.465862512588501
3.4602458477020264
3.4546773433685303
3.449155569076538
3.4436802864074707
3.4382519721984863
3.432868719100952
3.4275307655334473
3.4222378730773926
3.41698956489563
3.411785125732422


In [ ]:
#neither the bigram table or thies network prduce the trainset ouputs for trainset inputs obvious how can it u fool 

In [41]:
#sampling
g = torch.Generator().manual_seed(123)
for _ in range(5):
    ix = 0
    name  = ""
    while True:
        xenc = F.one_hot( torch.tensor([ix]), num_classes= 27)
        logits =  xenc.float() @  W
        counts = logits.exp()
        p = counts / counts.sum(dim = 1, keepdim= True)
        ix = torch.multinomial(p, num_samples=1, replacement= True, generator= g).item()
        if ix == 0:
            break
        name += itoc[ix]
    print(name)

ayl
eeiey
viannalizyr
pidouyra
jon
